# Text Classification (Sentiment)

## Setup

### Import Dependencies

In [12]:
import pandas as pd
import os
import ipynbname
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

### Load CSV file into DataFrame

In [13]:
load_dotenv()

df = pd.read_csv("../data/transformed_news_data.csv")

df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_of_week
0,95586285,2026-05-13,Q1 Earnings Season Reconfirms the Positive Ear...,zacks.com,"('Consumer Cyclical', 'ETF', 'Earnings Trends'...",2026-05-13,"('amzn', 'googl', 'meta', 'meta', 'metv', 'msf...",3
1,95586286,2026-05-13,Q1 Earnings Season Reconfirms the Positive Ear...,zacks.com,"('Consumer Cyclical', 'ETF', 'Earnings Outlook...",2026-05-13,"('amzn', 'googl', 'meta', 'meta', 'metv', 'msf...",3
2,95582601,2026-05-13,Snap’s Q1 Makes Its AR Glasses Bet Harder To I...,forbes.com,"('Apple', 'Ar Glasses', 'Augmented Reality', '...",2026-05-13,"('goog', 'googl', 'meta', 'meta', 'metv', 'snap')",3
3,95584537,2026-05-13,"Top Analyst Reports for Meta, Pfizer & Salesforce",zacks.com,"('ETF', 'Energy', 'Healthcare', 'Industrials',...",2026-05-13,"('cop', 'idxx', 'meta', 'meta', 'metv', 'pfe',...",3
4,95576774,2026-05-13,"Meta Platforms, Inc. (META): Chris Rokos Trims...",insidermonkey.com,"('Stock', 'Technology')",2026-05-13,"('meta',)",3


## Text Classification (sentiment)

### Run model on news data

In [14]:
def run_text_classification(df):
    api_key = os.getenv('hugging_face_token')

    # Check if user has a hugging face api key
    if api_key is None:
        raise Exception('hugging_face_token environment variable not set in a .env file')

    client = InferenceClient(
        provider="hf-inference",
        api_key=api_key,
    )

    results = client.text_classification(
        df['title'].to_list(),
        model="ProsusAI/finbert",
    )

    # Generated with AI (Gemini 3 fast)
    df['sentiment_label'] = [res.label for res in results]
    df['sentiment_score'] = [res.score for res in results]
    return df

df = run_text_classification(df)
df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_of_week,sentiment_label,sentiment_score
0,95586285,2026-05-13,Q1 Earnings Season Reconfirms the Positive Ear...,zacks.com,"('Consumer Cyclical', 'ETF', 'Earnings Trends'...",2026-05-13,"('amzn', 'googl', 'meta', 'meta', 'metv', 'msf...",3,positive,0.910511
1,95586286,2026-05-13,Q1 Earnings Season Reconfirms the Positive Ear...,zacks.com,"('Consumer Cyclical', 'ETF', 'Earnings Outlook...",2026-05-13,"('amzn', 'googl', 'meta', 'meta', 'metv', 'msf...",3,positive,0.910511
2,95582601,2026-05-13,Snap’s Q1 Makes Its AR Glasses Bet Harder To I...,forbes.com,"('Apple', 'Ar Glasses', 'Augmented Reality', '...",2026-05-13,"('goog', 'googl', 'meta', 'meta', 'metv', 'snap')",3,negative,0.721280
3,95584537,2026-05-13,"Top Analyst Reports for Meta, Pfizer & Salesforce",zacks.com,"('ETF', 'Energy', 'Healthcare', 'Industrials',...",2026-05-13,"('cop', 'idxx', 'meta', 'meta', 'metv', 'pfe',...",3,neutral,0.938416
4,95576774,2026-05-13,"Meta Platforms, Inc. (META): Chris Rokos Trims...",insidermonkey.com,"('Stock', 'Technology')",2026-05-13,"('meta',)",3,neutral,0.895403


In [5]:
def enrich_df(df):
    # Assign numerical labels to each sentiment
    df['numerical_sentiment'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'positive'
                                                     else (0 if x['sentiment_label'] == 'neutral' else -1),
                                                     axis=1)

    # Flag if an article is positive, negative or neutral
    df['is_positive'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'positive' else 0, axis=1)
    df['is_negative'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'negative' else 0, axis=1)
    df['is_neutral'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'neutral' else 0, axis=1)

    return df

df = enrich_df(df)
df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_of_week,sentiment_label,sentiment_score,numerical_sentiment,is_positive,is_negative,is_neutral
0,95586285,2026-05-13,Q1 Earnings Season Reconfirms the Positive Ear...,zacks.com,"('Consumer Cyclical', 'ETF', 'Earnings Trends'...",2026-05-13,"('amzn', 'googl', 'meta', 'meta', 'metv', 'msf...",3,positive,0.910511,1,1,0,0
1,95586286,2026-05-13,Q1 Earnings Season Reconfirms the Positive Ear...,zacks.com,"('Consumer Cyclical', 'ETF', 'Earnings Outlook...",2026-05-13,"('amzn', 'googl', 'meta', 'meta', 'metv', 'msf...",3,positive,0.910511,1,1,0,0
2,95582601,2026-05-13,Snap’s Q1 Makes Its AR Glasses Bet Harder To I...,forbes.com,"('Apple', 'Ar Glasses', 'Augmented Reality', '...",2026-05-13,"('goog', 'googl', 'meta', 'meta', 'metv', 'snap')",3,negative,0.721280,-1,0,1,0
3,95584537,2026-05-13,"Top Analyst Reports for Meta, Pfizer & Salesforce",zacks.com,"('ETF', 'Energy', 'Healthcare', 'Industrials',...",2026-05-13,"('cop', 'idxx', 'meta', 'meta', 'metv', 'pfe',...",3,neutral,0.938416,0,0,0,1
4,95576774,2026-05-13,"Meta Platforms, Inc. (META): Chris Rokos Trims...",insidermonkey.com,"('Stock', 'Technology')",2026-05-13,"('meta',)",3,neutral,0.895403,0,0,0,1


### Calculate Metrics

In [17]:
def calculate_metrics(df):
    # Calculate metrics
    metrics_df = df.groupby(['published_date'])\
                    .agg({
                    'numerical_sentiment': 'mean',
                    'sentiment_score': 'mean',
                    'is_positive': 'sum',
                    'is_negative': 'sum',
                    'is_neutral': 'sum',
                    'sentiment_label': 'count'})\
                    .reset_index()\
                    .rename(
                    columns={'published_date': 'date',
                             'is_positive': 'positive_count',
                             'is_negative': 'negative_count',
                             'is_neutral': 'neutral_count',
                             'sentiment_label': 'total_articles',
                             'numerical': 'mean_sentiment',
                             'sentiment_score': 'mean_sentiment_probability'})

    prefix = ''
    # Allows function to be run within notebook or when called by external files
    if ipynbname.name() == 'sentiment_model':
        prefix = '../'

    file_name = f'{prefix}data/sentiment_counts.csv'

    metrics_df.to_csv(file_name, mode='a', index=False, header=None)
    sentiment_counts_df = pd.read_csv(file_name).drop_duplicates(subset=['date'], keep='first')
    sentiment_counts_df.to_csv(file_name, index=False)

    # Calculate Percentages
    metrics_df['percent_positive'] = metrics_df['positive_count'] / metrics_df['total_articles']
    metrics_df['percent_negative'] = metrics_df['negative_count'] / metrics_df['total_articles']
    metrics_df['percent_neutral'] = metrics_df['neutral_count'] / metrics_df['total_articles']

    # Remove count columns use the percentages of totals instead
    metrics_df = metrics_df.drop(['positive_count', 'negative_count', 'neutral_count', 'total_articles'], axis=1)

    return metrics_df

metrics_df = calculate_metrics(df)
metrics_df.head()

sentiment_model


,date,numerical_sentiment,mean_sentiment_probability,percent_positive,percent_negative,percent_neutral
0,2026-01-28,0.246914,0.801551,0.345679,0.098765,0.555556
1,2026-01-29,-0.027027,0.806134,0.209459,0.236486,0.554054
2,2026-01-30,-0.109375,0.837781,0.156250,0.265625,0.578125
3,2026-01-31,0.117647,0.832265,0.147059,0.029412,0.823529
4,2026-02-01,0.000000,0.837922,0.230769,0.230769,0.538462


## Write output DataFrame as CSV File

In [18]:
# Output as CSV
metrics_df.to_csv('../data/article_metrics.csv', index=False)